### 🧰 CodeCritic Linting Dataset Generator — Test Corpus, Execution, and Backup

This notebook builds a **reproducible dataset** for evaluating CodeCritic’s linting system. It generates a synthetic set of Python files with diverse code issues, runs them through the full program pipeline, and captures detailed provider logs for downstream analysis.

#### 🔍 Overview

- **📁 Test File Generation**  
  Creates 10 Python scripts with controlled variations in formatting, logic, type hints, and structure. These files simulate realistic code quality issues for CodeCritic to lint and score.

- **🚀 Batch Execution via Program Provider**  
  Iterates through each test file using a preconfigured session and invokes the `ProgramProvider` for the `linting` system. Results are printed and logged to the database.

- **💾 Provider Log Backup to CSV**  
  Backs up the `provider_log` table after execution to a persistent CSV file. This snapshot can be restored or reused in future notebooks for evaluation and benchmarking.

- **📂 Restore Preview and Verification**  
  Loads and displays the first rows of the backup to validate structure and inspect output consistency.

#### 🧠 Why this matters

This notebook establishes a **baseline dataset** for EDA, scoring diagnostics, and visualization across linting runs. By standardizing input files and saving output logs, it ensures **repeatable, consistent** analyses for system tuning and evaluation.


### 🛠️ Project Root Setup + Database Reset and Seeding

This cell resets the entire database schema and seeds it with initial data. It is typically used in local development or test environments before running sessions.

#### 🧠 What this code does:

1. **Sets the working directory** to the project root (`C:\Repos\codecritic`) so all relative imports and paths resolve correctly.
2. **Imports all seeders** from `app.db.seeders` to populate the database with provider and configuration records.
3. **Drops and recreates** all tables using SQLAlchemy’s `Base.metadata.drop_all()` and `create_all()` for a clean start.
4. **Runs all seeders** in a single transaction using `Session(bind=engine)` to insert:
   - Prompts and prompt providers
   - Tools, scores, and agent engines
   - Agents, states, systems, controllers, programs, and sessions
5. **Loads environment variables** from `env/.env` relative to the project root.

#### 🔍 Why it matters:

- Ensures the system starts from a **clean and reproducible state**
- Guarantees that all required provider records are seeded before executing sessions
- Makes local development or testing **deterministic**
- Prepares the environment for any downstream scripts, agents, or workflows

Use this cell at the start of notebooks that need access to a freshly initialized and fully seeded CodeCritic environment.

In [1]:
from pathlib import Path
import shutil
import os, sys
from dotenv import load_dotenv

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

# Define the directories to be emptied
directories_to_clear = [
    PROJECT_ROOT / "working_files",
    PROJECT_ROOT / "extensions",
    PROJECT_ROOT / "experiments" / "snapshots"
]

# Function to delete all files and directories within a specified directory
def clear_directory(directory: Path):
    if directory.exists() and directory.is_dir():
        for item in directory.iterdir():
            try:
                if item.is_dir():
                    shutil.rmtree(item)  # Remove directory and all its contents
                else:
                    item.unlink()  # Remove file
            except Exception as e:
                print(f"Error deleting {item}: {e}")

# Clear all the directories
for directory in directories_to_clear:
    clear_directory(directory)

print("All specified directories have been cleared.")

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_score_providers import seed_score_providers
from app.db.seeders.seed_tool_providers import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.seeders.seed_system_providers import seed_system_providers
from app.db.seeders.seed_controller_providers import seed_controller_providers
from app.db.seeders.seed_program_provider import seed_program_providers
from app.db.seeders.seed_session_configs import seed_session_configs
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)
    seed_system_providers(session)
    seed_controller_providers(session)
    seed_program_providers(session)
    seed_session_configs(session)

# Load from env/.env relative to project root
dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")


Working directory is now: C:\Repos\codecritic
All specified directories have been cleared.
Seeded AgentPrompt 'generate' with ID: 1 and GUID: 3967bcf0-5b96-4797-a279-eaaa811eef41
Seeded AgentPrompt 'linting_generator_agent' with ID: 2 and GUID: dee70898-3d4b-487c-ab9f-de038c412e87
Seeded SystemPrompt 'format' with ID: 1 and GUID: 606f6e6c-88ad-44c8-9339-31f3aa1d6b3c
Seeded SystemPrompt 'linting_system' with ID: 2 and GUID: 0f9a3ed5-67e9-4ebc-abcc-d307cf0902fe
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
✅ Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
✅ Seeded state provider configurations successfully.
✅ Seeded system provider configurations successfully.
✅ Seeded controller providers
✅ Seeded program providers
✅ Seeded session configs
✅ Loaded environment variables from env/.env


### 🧪 Test File Generator — Linting Evaluation Corpus

This cell creates a synthetic set of 10 Python files under the `tests/test_files/` directory, each designed to trigger different linting behaviors in the CodeCritic pipeline.

#### 🧠 What this code does:

1. **Creates the test directory** `tests/test_files` if it doesn't exist.
2. **Defines 10 unique Python scripts**, each introducing distinct code smells or patterns such as:
   - Valid clean code
   - Unused imports
   - Poor indentation or mixing of tabs/spaces
   - Missing type hints and docstrings
   - Off-by-one logic errors
   - Built-in name shadowing
   - Long lines and structural complexity
   - Incorrect type annotations

3. **Writes each script to disk** with a meaningful filename that reflects the issue it demonstrates.

#### 🔍 Why it matters:

- Provides a **reproducible test corpus** for evaluating linting quality, model decisions, and score transitions.
- Enables batch analysis across files with **controlled variability** in error types.
- Supports visualizations and benchmarking in downstream notebooks.
- Makes it easy to reset and rerun experiments without requiring real project files.

Use this cell once at the start of a notebook or script that needs sample files to drive the CodeCritic linting system.


In [2]:
from pathlib import Path

TEST_FILES_DIR = Path("tests/test_files")
TEST_FILES_DIR.mkdir(parents=True, exist_ok=True)

examples = [
    # 1. Simple valid Python
    ("valid_simple.py", "def add(a, b):\n    return a + b\n"),

    # 2. Unused import + extra newline
    ("unused_import.py", "import os\n\nprint('Hello World')\n"),

    # 3. Bad indentation
    ("bad_indentation.py", "def greet():\nprint('Hello')\n"),

    # 4. Mixed tabs/spaces
    ("mixed_tabs_spaces.py", "def f():\n\treturn 42\n"),

    # 5. No function docstring or type hints
    ("no_doc_type.py", "def compute(x,y):\n return x*y\n"),

    # 6. Logic bug
    ("off_by_one.py", "def inclusive_range(n):\n    return list(range(n))  # should be range(n+1)\n"),

    # 7. Shadowing built-ins
    ("shadow_builtin.py", "list = [1, 2, 3]\nprint(list)\n"),

    # 8. Overly complex function
    ("complex_function.py", "def decision(x):\n if x>0: return 'yes'\n elif x<0: return 'no'\n else: return 'maybe'\n"),

    # 9. Long lines
    ("long_line.py", "def long(): print('This is a very long line that should probably be split over multiple lines for readability purposes')\n"),

    # 10. Type hint issues
    ("wrong_type_hint.py", "def square(x: str) -> int:\n    return x * x\n")
]

for fname, content in examples:
    (TEST_FILES_DIR / fname).write_text(content)


### 🚀 Batch Program Execution — Linting Across Test Files

This cell runs the CodeCritic linting program on every file in the `tests/test_files/` directory, invoking the configured session and logging results for each.

#### 🧠 What this code does:

1. **Loads the active session configuration** from the `SessionConfig` table (`id=1`) to retrieve the associated `program_provider_id`.
2. **Creates a `ProgramProvider` instance** using the session's configuration, simulating a top-level call from the SESSION layer.
3. **Iterates through all test files** in the `tests/test_files/` directory, sorted by name.
4. **Prepares input data** for each file with:
   - File path
   - Session ID
   - System type (`linting`)
5. **Runs the program provider** on each file and prints the structured result.

#### 🔍 Why it matters:

- Ensures **uniform execution** of the CodeCritic pipeline on a consistent test corpus.
- Simulates a real-world session-driven workflow, preserving traceability and reproducibility.
- Generates a **rich provider log trail** that can be analyzed across files for scoring, latency, and transformation effectiveness.
- Provides the core dataset used in subsequent analysis and visualization notebooks.

Run this cell after generating the test files to populate the database with fresh linting execution logs.


In [3]:
# 🧪 Run Program Using SessionConfig for All Test Files

from app.db.models import SessionConfig
from app.enums.logging_enums import PROVIDER_TYPE, RunContext
from app.enums.system_enums import SYSTEM
from app.factories.program_provider_factory import ProgramProviderFactory
from sqlalchemy.orm import Session
from pathlib import Path
import json

TEST_DIR = Path("tests/test_files")

# Fetch session config
with Session(bind=engine) as db:
    session_row = db.query(SessionConfig).filter_by(id=1).first()
    assert session_row, "❌ No session config found"

# Iterate through test files
for file in sorted(TEST_DIR.glob("*.py")):
    input_data = {
        "file_path":  str(file),
        "session_id": session_row.id,
        "system":     SYSTEM.LINTING.value,
    }

    context = RunContext(
        called_by_type=PROVIDER_TYPE.SESSION,
        called_by_id=session_row.id,
        session_id=session_row.id,
        file_log_id=None,
        execution_chain=[]
    )

    program = ProgramProviderFactory.create(
        id=session_row.program_provider_id,
        context=context
    )

    print(f"\n📂 Running program for file: {file.name}")
    result = program.run(input_data, context=context)
    print(json.dumps(result.model_dump(), indent=2))



📂 Running program for file: bad_indentation.py
{
  "state": "end",
  "previous_state": "preprocessing_controller",
  "state_type": "end",
  "decision": "accepted",
  "steps": 2,
  "max_steps": 20,
  "summary": "completed successfully",
  "output": {
    "state": "end",
    "file_path": "working_files\\bad_indentation.py",
    "session_id": 1,
    "file_log_id": 1,
    "reason": "completed successfully",
    "steps": 2,
    "retry_count": 0,
    "_last_state": "preprocessing_controller",
    "decision": "accepted",
    "original_file": "tests\\test_files\\bad_indentation.py",
    "run_id": "5639f147-aecc-491a-b0cc-e50e2ea630f8",
    "transition_metadata": {
      "original_file": "tests\\test_files\\bad_indentation.py",
      "run_id": "5639f147-aecc-491a-b0cc-e50e2ea630f8",
      "agent_output": {},
      "file_path": null
    },
    "state_output": {
      "state": "end",
      "previous_state": "linting",
      "state_type": "end",
      "decision": "accepted",
      "steps": 3,
   

### 💾 Provider Log Backup — CSV Export for Reuse

This cell exports the full contents of the `provider_log` table to a CSV file at `tests/backup/provider_logs.csv`, enabling reproducible analysis across notebooks and test runs.

#### 🧠 What this code does:

1. **Connects to the database** using SQLAlchemy’s session context.
2. **Queries all rows** from the `ProviderLog` table — including metadata like input, output, latency, file name, and config hashes.
3. **Converts each row** to a dictionary using the table’s column definitions (ORM-safe method).
4. **Builds a DataFrame** from the records and writes it to disk as a CSV file.
5. **Creates parent directories** (`tests/backup/`) if needed.

#### 🔍 Why it matters:

- Enables **quick snapshotting** of provider logs for reuse without rerunning the full pipeline.
- Makes downstream EDA notebooks **deterministic and portable** — they can operate on the same dataset repeatedly.
- Facilitates **versioned backups** or dataset sharing for debugging, regression testing, or performance benchmarking.
- Keeps backup logic consistent with the actual schema, avoiding Pydantic or serialization mismatches.

Use this cell after a test run to capture its full provider-level trace for analysis or restoration.


In [4]:
from sqlalchemy.orm import Session
from app.db.models import ProviderLog
import pandas as pd
from pathlib import Path

BACKUP_PATH = Path("tests/backup/provider_logs.csv")
BACKUP_PATH.parent.mkdir(parents=True, exist_ok=True)

with Session(bind=engine) as db:
    logs = db.query(ProviderLog).all()
    # SQLAlchemy-safe conversion
    df = pd.DataFrame([{
        column.name: getattr(log, column.name)
        for column in ProviderLog.__table__.columns
    } for log in logs])

    df.to_csv(BACKUP_PATH, index=False)
    print(f"✅ Backed up {len(df)} provider logs to {BACKUP_PATH}")


✅ Backed up 730 provider logs to tests\backup\provider_logs.csv


### 📂 Provider Log Preview — Load & Inspect Backup CSV

This cell loads the previously exported provider log backup from `tests/backup/provider_logs.csv` and displays the first 10 rows for inspection.

#### 🧠 What this code does:

1. **Validates existence** of the backup file before proceeding.
2. **Reads the CSV file** into a pandas DataFrame using `pd.read_csv()`.
3. **Displays the top 10 rows** using `df.head(10)` to verify data integrity and structure.

#### 🔍 Why it matters:

- Confirms the **backup was successful and readable** before attempting any restoration or analysis.
- Provides a **quick visual inspection** of key fields such as:
  - `provider_type`, `called_by_type`
  - `file_name`, `latency_ms`, `input`, `output`
  - `timestamp`, `config_hash`
- Ensures that all expected fields are present and correctly formatted.
- Acts as a lightweight **data sanity check** before running reloads, visualizations, or filtering logic.

Use this cell after backup to verify data integrity or before restore to confirm structure matches expectations.


In [5]:
import pandas as pd
from pathlib import Path

BACKUP_PATH = Path("tests/backup/provider_logs.csv")
assert BACKUP_PATH.exists(), "❌ Backup file not found"

df = pd.read_csv(BACKUP_PATH)

df.head(10)


,id,session_id,timestamp,provider_id,provider_type,input,output,output_schema,latency_ms,config_hash,file_name,called_by_type,called_by_id,run_id,file_log_id,parent_id,execution_chain
0,1,1,2025-06-16 18:48:23.196520+00:00,3,PROVIDER_TYPE.TOOL,"{""target"": ""C:\\Repos\\codecritic\\working_fil...","{""return_code"": 0, ""stdout"": ""[\n {\n \""ce...",ToolOutputSchema,27,99914b932bd37a50b983c5e7c90ae93b,af011cca-c192-4804-823c-8f2e9973aa33.py,PROVIDER_TYPE.SESSION,1,ac3fd372-36df-4ad2-8f6c-7c00dab16e87,1,c2ee0959-786d-4186-9da7-728bd1b02767,"['5639f147-aecc-491a-b0cc-e50e2ea630f8', '8fdf..."
1,2,1,2025-06-16 18:48:23.234802+00:00,1,PROVIDER_TYPE.TOOL,"{""target"": ""C:\\Repos\\codecritic\\working_fil...","{""return_code"": 0, ""stdout"": null, ""stderr"": ""...",ToolOutputSchema,223,99914b932bd37a50b983c5e7c90ae93b,7c47ebd6-8b87-4938-baae-5f6904d0cd4f.py,PROVIDER_TYPE.SESSION,1,6befdbc6-9e6c-4e9f-b9cb-ef0729639043,1,c2ee0959-786d-4186-9da7-728bd1b02767,"['5639f147-aecc-491a-b0cc-e50e2ea630f8', '8fdf..."
2,3,1,2025-06-16 18:48:23.463392+00:00,5,PROVIDER_TYPE.TOOL,"{""target"": ""C:\\Repos\\codecritic\\working_fil...","{""return_code"": 0, ""stdout"": null, ""stderr"": ""...",ToolOutputSchema,229,99914b932bd37a50b983c5e7c90ae93b,b80e70e0-14ba-4506-b705-e66edf4f8e3d.py,PROVIDER_TYPE.SESSION,1,2b5f99a3-cd03-4e68-a199-32e245c68e58,1,c2ee0959-786d-4186-9da7-728bd1b02767,"['5639f147-aecc-491a-b0cc-e50e2ea630f8', '8fdf..."
3,4,1,2025-06-16 18:48:23.196520+00:00,1,PROVIDER_TYPE.SCORE,"{""file_path"": ""C:\\Repos\\codecritic\\working_...","{""name"": ""linting_score"", ""value"": 0.9, ""compo...",ScoreOutputSchema,504,3b3085ec878f73d63ab3fd329a8756f2,13445310-005d-4041-851d-e01b881d2f12.py,PROVIDER_TYPE.SESSION,1,c2ee0959-786d-4186-9da7-728bd1b02767,1,728604f4-9725-4d24-ae5a-387d7b4b94d5,"['5639f147-aecc-491a-b0cc-e50e2ea630f8', '8fdf..."
4,5,1,2025-06-16 18:48:23.196520+00:00,2,PROVIDER_TYPE.CONTEXT,"{""file_path"": ""C:\\Repos\\codecritic\\working_...","{""context"": {""file_path"": ""C:\\Repos\\codecrit...",ContextOutputSchema,511,dbda5045e047c4f29f679f281b6021a7,d516111c-8b3d-4170-b68a-112fbaeacc7a.py,PROVIDER_TYPE.SESSION,1,728604f4-9725-4d24-ae5a-387d7b4b94d5,1,3964d7eb-6566-40e3-b708-422ee8ebc25b,"['5639f147-aecc-491a-b0cc-e50e2ea630f8', '8fdf..."
5,6,1,2025-06-16 18:48:23.196520+00:00,2,PROVIDER_TYPE.PROMPT,"{""file_path"": ""C:\\Repos\\codecritic\\working_...","{""prompt"": ""You are operating within the Linti...",PromptOutputSchema,517,e4f191e6df2d1e1b09753f09147ba721,a3a7dc11-15aa-4df7-9d00-6db30b0eacd3.py,PROVIDER_TYPE.SESSION,1,3964d7eb-6566-40e3-b708-422ee8ebc25b,1,d682041f-14b4-4163-8b25-e8a157a93df5,"['5639f147-aecc-491a-b0cc-e50e2ea630f8', '8fdf..."
6,7,1,2025-06-16 18:48:23.719221+00:00,2,PROVIDER_TYPE.AGENT_ENGINE,"{""prompt"": {""prompt"": ""You are operating withi...","{""response"": ""[CODE]\ndef greet():\n print(...",AgentEngineOutput,3598,99914b932bd37a50b983c5e7c90ae93b,C:\Repos\codecritic\extensions\2074cfb0-c8ea-4...,PROVIDER_TYPE.SESSION,1,5d577501-798b-4b42-b6a1-d1f4a7379fd5,1,d682041f-14b4-4163-8b25-e8a157a93df5,"['5639f147-aecc-491a-b0cc-e50e2ea630f8', '8fdf..."
7,8,1,2025-06-16 18:48:23.196520+00:00,2,PROVIDER_TYPE.AGENT,"{""file_path"": ""C:\\Repos\\codecritic\\working_...","{""agent_type"": ""generator"", ""decision"": ""unkno...",AgentOutputSchema,4132,4114db3a6b1b3b248a09bb2fcee90284,728aa2b2-1762-427b-be73-ad19f7f691c3.py,PROVIDER_TYPE.SESSION,1,d682041f-14b4-4163-8b25-e8a157a93df5,1,4ca3b203-6976-42bd-a801-d9cd30324688,"['5639f147-aecc-491a-b0cc-e50e2ea630f8', '8fdf..."
8,9,1,2025-06-16 18:48:27.331594+00:00,3,PROVIDER_TYPE.TOOL,"{""target"": ""C:\\Repos\\codecritic\\working_fil...","{""return_code"": 0, ""stdout"": ""[\n {\n \""ce...",ToolOutputSchema,26,99914b932bd37a50b983c5e7c90ae93b,af011cca-c192-4804-823c-8f2e9973aa33.py,PROVIDER_TYPE.SESSION,1,c8dd312b-69ca-4580-a605-1d94dee098d5,1,2caa7695-6e3c-4c83-ab99-dcac41f4a148,"['5639f147-aecc-491a-b0cc-e50e2ea630f8', '8fdf..."
9,10,1,2025-06-16 18:48: